In [1]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline

from xgboost import XGBRegressor

In [ ]:
TARGETS = [
    "Tar_mg_per_cigarette",
    "CO_mg_per_cigarette",
    "Phenol_ug_per_cigarette",
]

PUFF_GROUPS = ["ConeAir", "PaperAir", "VentAir"]

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
test_key = pd.read_csv(TEST_KEY_PATH)

BASE_FEATURES = [c for c in train.columns if c not in ["SampleID"] + TARGETS]

train.shape, test.shape, test_key.shape

((695, 40), (190, 37), (190, 5))

In [5]:
train[TARGETS].isna().sum()

Tar_mg_per_cigarette         0
CO_mg_per_cigarette          0
Phenol_ug_per_cigarette    390
dtype: int64

In [6]:
class PuffFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = pd.DataFrame(X).copy()

        for prefix in PUFF_GROUPS:
            cols = [f"{prefix}_puff_{i:02d}_mL" for i in range(1, 11)]
            vals = df[cols]

            df[f"{prefix}_sum"] = vals.sum(axis=1)
            df[f"{prefix}_mean"] = vals.mean(axis=1)
            df[f"{prefix}_std"] = vals.std(axis=1)
            df[f"{prefix}_max"] = vals.max(axis=1)
            df[f"{prefix}_last_nonzero"] = vals.mask(vals.eq(0)).ffill(axis=1).iloc[:, -1].fillna(0)
            df[f"{prefix}_nonzero_count"] = (vals > 0).sum(axis=1)
            df[f"{prefix}_slope_1_6"] = vals.iloc[:, 5].to_numpy() - vals.iloc[:, 0].to_numpy()

        df["TotalAir_sum"] = df[[f"{p}_sum" for p in PUFF_GROUPS]].sum(axis=1)
        df["Vent_ratio"] = df["VentAir_sum"] / (df["TotalAir_sum"] + 1e-6)
        df["Cone_ratio"] = df["ConeAir_sum"] / (df["TotalAir_sum"] + 1e-6)
        df["Paper_ratio"] = df["PaperAir_sum"] / (df["TotalAir_sum"] + 1e-6)
        df["Resistance_total"] = df["TobaccoRodResistance_Pa_cm"] + df["FilterResistance_Pa_cm"]
        df["Vent_span"] = df["FilterAfterVent_cm"] - df["FilterBeforeVent_cm"]
        df["Air_per_puff"] = df["TotalAir_sum"] / (df["NumPuffs"] + 1e-6)

        return df


In [7]:
def build_xgb_log_pipeline():
    regressor = Pipeline([
        ("features", PuffFeatureEngineer()),
        ("imputer", SimpleImputer(strategy="median")),
        (
            "xgb",
            XGBRegressor(
                n_estimators=900,
                max_depth=4,
                learning_rate=0.015,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_lambda=1.0,
                objective="reg:squarederror",
                random_state=42,
                n_jobs=4,
            ),
        ),
    ])

    return TransformedTargetRegressor(
        regressor=regressor,
        func=np.log1p,
        inverse_func=np.expm1,
    )


cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [8]:
results = []
predictions = {}

X_train = train[BASE_FEATURES]
X_test = test[BASE_FEATURES]

for target in TARGETS:
    mask = train[target].notna()
    Xtr = X_train.loc[mask]
    ytr = train.loc[mask, target]
    yte = test_key[target]

    model = build_xgb_log_pipeline()
    cv_scores = cross_val_score(model, Xtr, ytr, cv=cv, scoring="r2", n_jobs=1)
    model.fit(Xtr, ytr)
    pred = model.predict(X_test)

    predictions[target] = pred
    results.append({
        "target": target,
        "train_rows": int(mask.sum()),
        "cv_r2_mean": float(cv_scores.mean()),
        "cv_r2_std": float(cv_scores.std()),
        "test_r2": float(r2_score(yte, pred)),
    })

results_df = pd.DataFrame(results)
results_df

,target,train_rows,cv_r2_mean,cv_r2_std,test_r2
0,Tar_mg_per_cigarette,695,0.959916,0.009752,0.787150
1,CO_mg_per_cigarette,695,0.949973,0.011944,0.680352
2,Phenol_ug_per_cigarette,305,0.918994,0.029463,-0.274215


In [9]:
pred_df = pd.DataFrame(predictions)
pred_df.index = test["SampleID"]
pred_df.head()

,Tar_mg_per_cigarette,CO_mg_per_cigarette,Phenol_ug_per_cigarette
SampleID,,,
TEST-001,8.919521,8.250233,9.767251
TEST-002,10.647546,9.731287,12.965790
TEST-003,9.522285,8.711080,11.980017
TEST-004,8.613111,7.828513,9.239250
TEST-005,9.895475,9.303875,11.531542


In [10]:
avg_r2 = r2_score(test_key[TARGETS], pred_df.reset_index(drop=True)[TARGETS], multioutput="uniform_average")

for target in TARGETS:
    score = r2_score(test_key[target], pred_df[target])
    print(f"  {target}: {score:.6f}")
print(f"  Average R^2: {avg_r2:.6f}")

  Tar_mg_per_cigarette: 0.787150
  CO_mg_per_cigarette: 0.680352
  Phenol_ug_per_cigarette: -0.274215
  Average R^2: 0.397762


In [11]:
submission_like = pd.DataFrame({"SampleID": test["SampleID"]})
for target in TARGETS:
    submission_like[target] = pred_df[target].to_numpy()

submission_like.head()

,SampleID,Tar_mg_per_cigarette,CO_mg_per_cigarette,Phenol_ug_per_cigarette
0,TEST-001,8.919521,8.250233,9.767251
1,TEST-002,10.647546,9.731287,12.965790
2,TEST-003,9.522285,8.711080,11.980017
3,TEST-004,8.613111,7.828513,9.239250
4,TEST-005,9.895475,9.303875,11.531542
